# Player を継承した自作方策（AI）の最小実装を示す

choose_command をオーバーライドし、「最も威力の高い技を選ぶ」という単純な
ヒューリスティックのプレイヤーを作る。
AI開発ユースケースの最初のステップとして、Player のカスタム方法だけに絞った例。

Google Colabで開いた場合は、まず次のセルで `jpoke` をインストールする
（ローカルで `pip install jpoke` 済みなら不要）。

In [ ]:
%pip install -q jpoke

In [ ]:
from __future__ import annotations

from jpoke import Battle, Player
from jpoke.enums import Command

In [ ]:
class StrongestMovePlayer(Player):
    """毎ターン、利用可能な技の中から最も威力の高い技を選ぶプレイヤー。"""

    def choose_command(self, battle: Battle) -> Command:
        """利用可能なコマンドの中から、最も威力の高い技コマンドを選ぶ。

        メガシンカ・テラスタルを伴う技コマンドも対象に含める。交代・わるあがき等の
        技以外のコマンドは最低優先度として扱い、通常は選ばれない。
        """
        commands = battle.get_available_commands(self)

        def move_power(command: Command) -> int:
            # is_type("move") と is_regular_move の違いは
            # docs/api/README.md の Command「インスタンスプロパティ・メソッド」節を参照
            if not command.is_type("move"):
                return -1
            move = battle.command_to_move(self, command)
            # 変化技のbase_powerがNoneになる点はdocs/api/README.mdのMove「主要属性」節を参照
            return move.base_power if move.is_attack else 0

        # key=move_power は「commands の各要素を move_power() に通した結果を
        # 比較キーにして最大の要素を選ぶ」という意味。ここでは威力が最大の
        # コマンドを選ぶ（move_power() 自体は変更しない）
        return max(commands, key=move_power)

In [ ]:
player1 = StrongestMovePlayer("StrongestMovePlayer")
player1.add_pokemon("ピカチュウ", move_names=["でんこうせっか", "かみなり", "なきごえ"])

player2 = Player("Player 2")
player2.add_pokemon("フシギダネ", move_names=["たいあたり"])

In [ ]:
battle = Battle(player1, player2, seed=1)
battle.play_out(max_turns=100)
winner = battle.winner
print(f"勝者: {winner.username if winner else '引き分け（ターン上限）'}")
battle.print_logs("all")
print("-" * 50)  # print_logs() の出力とその後のprint()を視覚的に区切る

In [ ]:
# StrongestMovePlayer は威力110の「かみなり」を選び続けるはず
# （威力40の「でんこうせっか」・変化技の「なきごえ」より優先される）。
# player1.team が対戦中に更新されない点は docs/api/README.md の
# Player「team属性の注意点」節を参照。対戦中の状態は battle.get_active() で取得する
active = battle.get_active(player1)
print(f"最終ターンの技: {active.last_move.name if active.last_move else None}")

試してみよう: move_power() に「相手のタイプ相性が悪い技は除外する」といった
条件を加えると、より賢い方策に発展させられる